# Qwen 3.5 4B через llama.cpp

1. Запусти первую ячейку — она поднимет `llama-server`.
2. После сообщения `Сервер готов` запускай ячейку с чатом.
3. В конце можно остановить сервер последней ячейкой.

> Ноутбук предполагает, что он лежит в корне проекта рядом с папками `llama.cpp` и `models`.

In [10]:
import subprocess
import time
from urllib.request import urlopen
from urllib.error import URLError

HEALTH_URL = "http://127.0.0.1:8080/health"

try:
    urlopen(HEALTH_URL, timeout=1)
    print("llama-server уже запущен")
except URLError:
    server = subprocess.Popen([
        r".\llama.cpp\llama-server.exe",
        "--model", r".\models\Qwen_Qwen3.5-4B-Q4_K_M.gguf",
        "--alias", "qwen3.5-4b",
        "--host", "127.0.0.1",
        "--port", "8080",
        "--ctx-size", "4096",
        "--threads", "4",
        "--threads-batch", "4",
        "--reasoning", "off",
    ])

    while True:
        if server.poll() is not None:
            raise RuntimeError(
                f"llama-server завершился с кодом {server.returncode}"
            )

        try:
            urlopen(HEALTH_URL, timeout=1)
            print("llama-server запущен и готов")
            break
        except URLError:
            time.sleep(1)

llama-server запущен и готов


In [4]:
from openai import OpenAI


client = OpenAI(
    base_url="http://127.0.0.1:8080/v1",
    api_key="not-needed",
    timeout=300.0,
)

print("Отправляю запрос модели...")

response = client.chat.completions.create(
    model="qwen3.5-4b",
    messages=[
        {
            "role": "system",
            "content": (
                "Ты преобразуешь неструктурированные данные "
                "в строго структурированный результат."
            ),
        },
        {
            "role": "user",
            "content": (
                "Преобразуй запись в JSON: "
                "дата 15.07.2026, артикул A-17, "
                "сумма 1 250,50 руб."
            ),
        },
    ],
    temperature=0.1,
    max_tokens=256,
)

text = response.choices[0].message.content

if text is None:
    raise RuntimeError("Модель вернула пустой ответ")

print("\nОтвет модели:\n")
print(text)

Отправляю запрос модели...

Ответ модели:

```json
{
  "date": "2026-07-15",
  "article": "A-17",
  "amount": 1250.50,
  "currency": "RUB"
}
```


In [5]:
text = response.choices[0].message.content

if text is None:
    raise RuntimeError("Модель вернула пустой ответ")

print(text)

timings = response.model_extra.get("timings", {})
usage = response.usage

print("\nСтатистика:")

print(f"Токены промпта:      {usage.prompt_tokens}")
print(f"Токены генерации:    {usage.completion_tokens}")
print(f"Всего токенов:       {usage.total_tokens}")

print(f"Время префилла:      {timings['prompt_ms'] / 1000:.2f} с")
print(f"Скорость префилла:   {timings['prompt_per_second']:.2f} ток/с")

print(f"Время генерации:     {timings['predicted_ms'] / 1000:.2f} с")
print(f"Скорость генерации:  {timings['predicted_per_second']:.2f} ток/с")

```json
{
  "date": "2026-07-15",
  "article": "A-17",
  "amount": 1250.50,
  "currency": "RUB"
}
```

Статистика:
Токены промпта:      74
Токены генерации:    60
Всего токенов:       134
Время префилла:      1.30 с
Скорость префилла:   56.89 ток/с
Время генерации:     4.79 с
Скорость генерации:  12.52 ток/с


## Остановка сервера

Выполни эту ячейку, когда модель больше не нужна.

In [11]:
server = globals().get("server")

if server is None:
    print("Нет ссылки на процесс сервера — останавливать нечего.")
elif server.poll() is not None:
    print(f"Сервер уже остановлен. Код завершения: {server.returncode}")
else:
    server.terminate()

    try:
        server.wait(timeout=10)
        print("Сервер успешно остановлен.")
    except subprocess.TimeoutExpired:
        server.kill()
        server.wait()
        print("Сервер не завершился вовремя и был принудительно остановлен.")

Сервер успешно остановлен.
